In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
langchain-chroma \
chromadb \
wikipedia-api

In [2]:
import os
import wikipediaapi
import chromadb
import langchain

print("✅ Imports successful")

✅ Imports successful


Mount Drive

In [3]:
from google.colab import drive

try:
    drive.mount('/content/drive')
    print(" Drive mounted")
except Exception as e:
    print(" Error:", e)

Mounted at /content/drive
 Drive mounted


Create Project Folders

In [4]:
PROJECT_DIR = "/content/drive/MyDrive/RAG_Internship"

WIKI_DIR = f"{PROJECT_DIR}/wiki_docs"
CHROMA_DIR = f"{PROJECT_DIR}/chroma_db"

try:
    os.makedirs(WIKI_DIR, exist_ok=True)
    os.makedirs(CHROMA_DIR, exist_ok=True)

    print(" Folders created successfully")

except Exception as e:
    print(" Error:", e)

 Folders created successfully


In [5]:
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-internship-project'
)

In [6]:
topics = [
    "Artificial intelligence",
    "Machine learning",
    "Deep learning",
    "Neural network",
    "Large language model",
    "Generative artificial intelligence",
    "Natural language processing",
    "Computer vision",
    "Reinforcement learning",
    "Retrieval-augmented generation"
]

In [7]:
try:

    for topic in topics:

        page = wiki.page(topic)

        if page.exists():

            filename = topic.replace(" ", "_") + ".txt"

            with open(
                f"{WIKI_DIR}/{filename}",
                "w",
                encoding="utf-8"
            ) as f:

                f.write(page.text)

            print(f" Saved: {filename}")

        else:
            print(f"⚠️ Page not found: {topic}")

except Exception as e:
    print(" Error:", e)

 Saved: Artificial_intelligence.txt
 Saved: Machine_learning.txt
 Saved: Deep_learning.txt
 Saved: Neural_network.txt
 Saved: Large_language_model.txt
 Saved: Generative_artificial_intelligence.txt
 Saved: Natural_language_processing.txt
 Saved: Computer_vision.txt
 Saved: Reinforcement_learning.txt
 Saved: Retrieval-augmented_generation.txt


In [8]:
try:

    files = os.listdir(WIKI_DIR)

    print("Documents:", len(files))

    for file in files:
        print(file)

except Exception as e:
    print(" Error:", e)

Documents: 10
Artificial_intelligence.txt
Machine_learning.txt
Deep_learning.txt
Neural_network.txt
Large_language_model.txt
Generative_artificial_intelligence.txt
Natural_language_processing.txt
Computer_vision.txt
Reinforcement_learning.txt
Retrieval-augmented_generation.txt


Load Documents

In [9]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

try:

    loader = DirectoryLoader(
        WIKI_DIR,
        glob="*.txt",
        loader_cls=TextLoader
    )

    documents = loader.load()

    print(" Documents loaded:", len(documents))

except Exception as e:
    print(" Error:", e)

/tmp/ipykernel_1290/1880345257.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


 Documents loaded: 10


In [10]:
print(documents[0].page_content[:1000])

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI rese

Chunk Documents

In [11]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,
    chunk_overlap=250
)

chunks = splitter.split_documents(documents)

print(" Total Chunks:", len(chunks))

 Total Chunks: 245


In [12]:
print("Documents:", len(documents))
print("Chunks:", len(chunks))
print(
    "Average chunks per document:",
    len(chunks) / len(documents)
)

Documents: 10
Chunks: 245
Average chunks per document: 24.5


In [14]:
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = (
    userdata.get("Gemini_key")
)

print("Gemini key configured")

Gemini key configured


Embeddings

In [15]:
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Embeddings done successfully")

Embeddings done successfully


create chromaDB

In [16]:
from langchain_chroma import Chroma
import time

persist_dir = CHROMA_DIR

vectorstore = None

batch_size = 20

Store Chunks to chromaDB

In [17]:
for i in range(0, len(chunks), batch_size):

    batch = chunks[i:i+batch_size]

    print(
        f"Processing chunks "
        f"{i+1} to "
        f"{min(i+batch_size, len(chunks))}"
    )

    if vectorstore is None:

        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            persist_directory=persist_dir,
            collection_name="wiki_rag"
        )

    else:

        vectorstore.add_documents(batch)

    print(" Batch stored")

    time.sleep(35)

print(" All chunks stored")

Processing chunks 1 to 20
 Batch stored
Processing chunks 21 to 40
 Batch stored
Processing chunks 41 to 60
 Batch stored
Processing chunks 61 to 80
 Batch stored
Processing chunks 81 to 100
 Batch stored
Processing chunks 101 to 120
 Batch stored
Processing chunks 121 to 140
 Batch stored
Processing chunks 141 to 160
 Batch stored
Processing chunks 161 to 180
 Batch stored
Processing chunks 181 to 200
 Batch stored
Processing chunks 201 to 220
 Batch stored
Processing chunks 221 to 240
 Batch stored
Processing chunks 241 to 245
 Batch stored
 All chunks stored


In [18]:
try:

    count = vectorstore._collection.count()

    print("Documents stored:", count)

except Exception as e:

    print("Error:", e)

Documents stored: 245
